# Huấn luyện Box Critic trên Kaggle (PointSnap 3D, lớp 3 của Box Verifier)

Chuẩn bị trên máy (xem `docs/box-critic.md`):

1. `pointsnap critic-data --scenes tuning --samples-per-box 20 --out critic-data`
2. Chép `scripts/train_critic_example.py` vào thư mục `critic-data/`.
3. Kaggle → **Datasets → New Dataset**, tải lên thư mục `critic-data/`, để **Private** (nuScenes không cho phân phối lại).

Trên Kaggle: tạo notebook từ file này, **Add Input** dataset vừa tạo, bật **GPU** (Settings → Accelerator), rồi chạy lần lượt.

In [ ]:
from pathlib import Path

# Thư mục dataset đã thêm vào notebook (đổi nếu đặt tên khác)
DATA = next(p.parent for p in Path('/kaggle/input').rglob('train_critic_example.py'))
print(DATA, sorted(p.name for p in DATA.glob('*.npz')))

In [ ]:
import numpy as np, torch
print('torch', torch.__version__, 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')
for p in sorted(DATA.glob('*.npz')):
    d = np.load(p)
    print(p.name, d['points'].shape, 'tỉ lệ mẫu sai', round(float(d['is_error'].mean()), 3))

Huấn luyện. Tăng `--epochs` / đổi `--lr` tùy kết quả; script in độ chính xác và sai số gợi ý dời trên tập kiểm tra mỗi epoch.

In [ ]:
!python {DATA}/train_critic_example.py --data {DATA} --epochs 40 --batch 512 --out /kaggle/working/critic.pt

Kiểm tra file đúng **hợp đồng** PointSnap: `logit, delta = model(points (B,256,4), meta (B,20))`.

In [ ]:
m = torch.jit.load('/kaggle/working/critic.pt', map_location='cpu').eval()
logit, delta = m(torch.zeros(2, 256, 4), torch.zeros(2, 20))
assert logit.shape == (2,) and delta.shape == (2, 7)
print('OK — tải /kaggle/working/critic.pt (tab Output) về máy, khai trong guideline:  critic: {model: <đường dẫn>/critic.pt}')